In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import os
import gc
import random
import warnings
import ctypes
import copy

In [2]:
def set_seed(seed: int) -> int:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2 ** 32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    print(f"Random seed: {seed}")
    return seed_worker

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

In [3]:
DATASET_CONFIG = {
    "appliances_energy": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/appliances_energy_dataset.csv", 
        "time_col": "date",
        # Để None để class tự động lấy toàn bộ các cột sensor (T1, RH_1,..., rv2) làm feature
        "feature_cols": [
            # Nhóm 1: Giá trị lịch sử của chính Target (rất quan trọng để mô hình học đà tiêu thụ)
            "Appliances", "lights",
            
            # Nhóm 2: Các cảm biến nhiệt độ (Temperature) trong và ngoài nhà
            "T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T_out",
            
            # Nhóm 3: Các cảm biến độ ẩm (Humidity) trong và ngoài nhà
            "RH_1", "RH_2", "RH_3", "RH_4", "RH_5", "RH_6", "RH_7", "RH_8", "RH_9", "RH_out",
            
            # Nhóm 4: Các thông số thời tiết khác
            "Press_mm_hg", "Windspeed", "Visibility", "Tdewpoint",
            
            # Nhóm 5: Feature Engineering thời gian (Bắt chu kỳ tiêu thụ điện)
            "hour_sin", "hour_cos", 
        ], 
        # Dự đoán 2 biến mục tiêu
        "target_cols": ["Appliances", "lights"]
    },

    "sunspots": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/sunspots_dataset.csv", 
        "time_col": "Date",
        # Loại bỏ cột index bị thừa trong file CSV
        "drop_cols": ["Unnamed: 0"],
        
        "feature_cols": [
            # Nhóm 1: Giá trị lịch sử của chính Target (Đây là dữ liệu univariate - đơn biến)
            "Monthly Mean Total Sunspot Number",
            
            # Nhóm 2: Feature Engineering thời gian (Chỉ giữ lại chu kỳ tháng)
            "month_sin", "month_cos"
        ], 
        # Dự đoán biến mục tiêu
        "target_cols": ["Monthly Mean Total Sunspot Number"]
    },

   "beijing_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/beijing_air_quality_dataset.csv",
        "time_col": ["year", "month", "day", "hour"],

        "feature_cols": [
            "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
            "TEMP", "PRES", "DEWP", "RAIN", "WSPM",

            "hour_sin", "hour_cos",
            "month_sin", "month_cos"
        ],

        "target_cols": [
            "PM2.5", "PM10", "SO2", "NO2", "CO", "O3"
        ]
    },

    "hanoi_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/hanoi_air_quality_dataset.csv",
        "time_col": "Local Time",
        
        # Danh sách các đặc trưng (loại bỏ các cột đã làm target để tránh rò rỉ dữ liệu, tùy thuộc vào logic code của bạn)
        "feature_cols": [
            "PM25", "PM10", "AQI", "CO", "NO2", "O3", "SO2", 
            "Clouds", "Precipitation", "Pressure", 
            "Relative Humidity", "Temperature", "UV Index", "Wind Speed",
            "hour_sin", "hour_cos",
            "month_sin", "month_cos"
        ],
        
        # Điền TẤT CẢ các cột mục tiêu vào đây
        "target_cols": ["PM25", "PM10", "AQI", "CO", "NO2", "O3", "SO2"] 
    },

   "bitcoin": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/bitcoin_dataset.csv", # Kiểm tra lại đường dẫn
        "time_col": "Timestamp",
        "feature_cols": [
            "Open"
        ],
        "target_cols": [
            "Open"
        ]
    }
}

In [4]:
class TSWindowDataset(Dataset):
    def __init__(self, feature_df, target_df, seq_len, pred_len):
        """
        feature_df: DataFrame chứa các cột feature
        target_df : DataFrame chứa các cột cần dự đoán
        """
        feature_df = feature_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        target_df = target_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
    
        feature_df = feature_df.ffill().bfill()
        target_df = target_df.ffill().bfill()

        self.X_data = torch.tensor(feature_df.to_numpy(dtype=np.float32), dtype=torch.float32)
        self.y_data = torch.tensor(target_df.to_numpy(dtype=np.float32), dtype=torch.float32)

        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.X_data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        X = self.X_data[idx : idx + self.seq_len]

        y_start = idx + self.seq_len
        y_end = y_start + self.pred_len
        y = self.y_data[y_start : y_end]
        
        return X, y

class TimeSeriesDataset:
    def __init__(
        self,
        name,
        seq_len=24,
        pred_len=1,
        train_ratio=0.7,
        val_ratio=0.2,
        batch_size=64,
        num_workers=0,
        worker_init_fn=None,
        generator=None,
    ):
        if name not in DATASET_CONFIG:
            raise ValueError(f"Tên dataset '{name}' không tồn tại trong DATASET_CONFIG.")

        self.name = name
        self.config = DATASET_CONFIG[name].copy()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.batch_size = batch_size

        self.num_workers = num_workers
        self.worker_init_fn = worker_init_fn
        self.generator = generator

        self.df_raw = None
        self.df_processed = None

        self.df_train = None
        self.df_val = None
        self.df_test = None

        self.X_train = None
        self.X_val = None
        self.X_test = None

        self.y_train = None
        self.y_val = None
        self.y_test = None

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

        self.feature_cols = None
        self.target_cols = None

        self.feature_scaler = None
        self.target_scaler = None
        self.is_scaled = False

        self._load_and_preprocess_data()
        self._split_data()
        self._create_datasets()

    def _load_and_preprocess_data(self):
        df = pd.read_csv(self.config["path"])
        self.df_raw = df.copy()

        time_col = self.config.get("time_col", None)
        
        if time_col is None:
            raise ValueError("Bạn cần khai báo 'time_col' trong DATASET_CONFIG.")

        # =========================
        # 1. Tạo cột datetime
        # =========================
        if isinstance(time_col, list):
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")
        else:
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")

        df = df.dropna(subset=["datetime"])
        df = df.sort_values("datetime").reset_index(drop=True)

        # =========================
        # 2. Loại bỏ các cột không cần thiết theo yêu cầu
        # =========================
        drop_cols = self.config.get("drop_cols", [])
        cols_to_drop = [col for col in drop_cols if col in df.columns]
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)

        # Giữ lại logic bỏ cột No/station cho tính tương thích ngược nếu bạn dùng lại data cũ
        if "No" in df.columns:
            df = df.drop(columns=["No"])
        if "station" in df.columns:
            df = df.drop(columns=["station"])

        # =========================
        # 3. Tạo feature thời gian dạng số (Giữ nguyên)
        # =========================
        df["hour"] = df["datetime"].dt.hour
        df["month"] = df["datetime"].dt.month
        df["dayofweek"] = df["datetime"].dt.dayofweek
        
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


        # =========================
        # 4. Xử lý Missing Values
        # =========================
        if "wd" in df.columns:
            if df["wd"].isna().sum() > 0:
                df["wd"] = df["wd"].fillna(df["wd"].mode()[0])
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[numeric_cols] = df[numeric_cols].ffill()
        df[numeric_cols] = df[numeric_cols].bfill()
        
        if "wd" in df.columns:
            df = pd.get_dummies(df, columns=["wd"], prefix="wd", dtype=np.float32)

        # =========================
        # 5. Xác định feature_cols và target_cols
        # =========================
        target_cols = self.config.get("target_cols", None)
        feature_cols = self.config.get("feature_cols", None)

        if target_cols is None:
            raise ValueError("Bạn cần khai báo 'target_cols' trong DATASET_CONFIG.")

        # =========================
        # 1. Tạo cột datetime
        # =========================
        if isinstance(time_col, list):
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")
        else:
            # Xử lý đặc thù cho file Bitcoin: convert Unix timestamp (giây) sang Datetime
            if self.name == "bitcoin":
                df["datetime"] = pd.to_datetime(df[time_col], unit='s', errors="coerce")
            else:
                df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")

        df = df.dropna(subset=["datetime"])
        df = df.sort_values("datetime").reset_index(drop=True)

        # TIẾN HÀNH RESAMPLE CHO BITCOIN
        if self.name == "bitcoin":
            df.set_index("datetime", inplace=True)
            # Gộp mỗi 1 giờ ('1h') và lấy giá trị chốt cuối cùng của giờ đó
            df = df.resample("1h").last()
            
            # Xóa các giờ không có giao dịch (dữ liệu rỗng)
            df = df.dropna(subset=["Open"]) 
            df.reset_index(inplace=True)

        if feature_cols is None:
            feature_cols = df.select_dtypes(include=[np.number, bool]).columns.tolist()
            feature_cols = [col for col in feature_cols if col != "datetime"]
        else:
            feature_cols = feature_cols.copy()

        wd_cols = [col for col in df.columns if col.startswith("wd_")]
        for col in wd_cols:
            if col not in feature_cols:
                feature_cols.append(col)

        missing_features = [col for col in feature_cols if col not in df.columns]
        missing_targets = [col for col in target_cols if col not in df.columns]

        if missing_features:
            raise ValueError(f"Các feature_cols không tồn tại trong DataFrame: {missing_features}")

        if missing_targets:
            raise ValueError(f"Các target_cols không tồn tại trong DataFrame: {missing_targets}")

        keep_cols = ["datetime"] + list(dict.fromkeys(feature_cols + target_cols))
        df = df[keep_cols]

        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.df_processed = df

    def _split_data(self):
        df = self.df_processed.copy()
    
        n = len(df)
        train_end = int(n * self.train_ratio)
        val_end = int(n * (self.train_ratio + self.val_ratio))
    
        self.df_train = df.iloc[:train_end].reset_index(drop=True)
        self.df_val = df.iloc[train_end:val_end].reset_index(drop=True)
        self.df_test = df.iloc[val_end:].reset_index(drop=True)
    
        self.X_train = self.df_train[self.feature_cols].copy()
        self.X_val = self.df_val[self.feature_cols].copy()
        self.X_test = self.df_test[self.feature_cols].copy()
    
        self.y_train = self.df_train[self.target_cols].copy()
        self.y_val = self.df_val[self.target_cols].copy()
        self.y_test = self.df_test[self.target_cols].copy()
    
        self.X_train = self.X_train.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        self.X_val = self.X_val.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        self.X_test = self.X_test.apply(pd.to_numeric, errors="coerce").astype(np.float32)
    
        self.y_train = self.y_train.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        self.y_val = self.y_val.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        self.y_test = self.y_test.apply(pd.to_numeric, errors="coerce").astype(np.float32)
    
        self.X_train = self.X_train.ffill().bfill()
        self.X_val = self.X_val.ffill().bfill()
        self.X_test = self.X_test.ffill().bfill()
    
        self.y_train = self.y_train.ffill().bfill()
        self.y_val = self.y_val.ffill().bfill()
        self.y_test = self.y_test.ffill().bfill()

    def apply_scaling(self, scaler_type="standard"):
        if scaler_type == "standard":
            self.feature_scaler = StandardScaler()
            self.target_scaler = StandardScaler()
        elif scaler_type == "minmax":
            self.feature_scaler = MinMaxScaler()
            self.target_scaler = MinMaxScaler()
        else:
            raise ValueError("scaler_type chỉ nhận 'standard' hoặc 'minmax'.")

        self.feature_scaler.fit(self.X_train)
        self.target_scaler.fit(self.y_train)

        self.X_train = pd.DataFrame(self.feature_scaler.transform(self.X_train), columns=self.feature_cols)
        self.X_val = pd.DataFrame(self.feature_scaler.transform(self.X_val), columns=self.feature_cols)
        self.X_test = pd.DataFrame(self.feature_scaler.transform(self.X_test), columns=self.feature_cols)

        self.y_train = pd.DataFrame(self.target_scaler.transform(self.y_train), columns=self.target_cols)
        self.y_val = pd.DataFrame(self.target_scaler.transform(self.y_val), columns=self.target_cols)
        self.y_test = pd.DataFrame(self.target_scaler.transform(self.y_test), columns=self.target_cols)

        self.is_scaled = True
        self._create_datasets()

    def inverse_transform_target(self, y_scaled):
        if self.target_scaler is None:
            raise ValueError("Bạn chưa gọi apply_scaling(), không có target_scaler để inverse transform.")

        if isinstance(y_scaled, torch.Tensor):
            y_scaled = y_scaled.detach().cpu().numpy()

        original_shape = y_scaled.shape
        y_2d = y_scaled.reshape(-1, len(self.target_cols))
        y_inv = self.target_scaler.inverse_transform(y_2d)

        return y_inv.reshape(original_shape)

    def _create_datasets(self):
        self.train_dataset = TSWindowDataset(self.X_train, self.y_train, self.seq_len, self.pred_len)
        self.val_dataset = TSWindowDataset(self.X_val, self.y_val, self.seq_len, self.pred_len)
        self.test_dataset = TSWindowDataset(self.X_test, self.y_test, self.seq_len, self.pred_len)

    def get_loaders(self):
        train_loader = DataLoader(
            self.train_dataset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, worker_init_fn=self.worker_init_fn, generator=self.generator
        )

        val_loader = DataLoader(
            self.val_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, worker_init_fn=self.worker_init_fn, generator=self.generator
        )

        test_loader = DataLoader(
            self.test_dataset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, worker_init_fn=self.worker_init_fn, generator=self.generator
        )

        return train_loader, val_loader, test_loader

    def print_info(self):
        X_tr, y_tr = self.train_dataset[0]

        train_batches = len(self.train_dataset) // self.batch_size
        if len(self.train_dataset) % self.batch_size != 0: train_batches += 1

        val_batches = len(self.val_dataset) // self.batch_size
        if len(self.val_dataset) % self.batch_size != 0: val_batches += 1

        test_batches = len(self.test_dataset) // self.batch_size
        if len(self.test_dataset) % self.batch_size != 0: test_batches += 1

        print(f"=== THÔNG TIN DATASET: {self.name.upper()} ===")
        print(f"- File path: {self.config['path']}")
        print(f"- Số dòng sau xử lý: {len(self.df_processed)}")
        print(f"- Chuẩn hóa: {'Có' if self.is_scaled else 'Không'}")
        print(f"- seq_len: {self.seq_len}")
        print(f"- pred_len: {self.pred_len}")
        print(f"- batch_size: {self.batch_size}")
        print(f"- num_workers: {self.num_workers}")
        print("-" * 60)

        print(f"- Số feature: {len(self.feature_cols)}")
        print(f"- Feature columns:")
        print(self.feature_cols)

        print("-" * 60)

        print(f"- Số target: {len(self.target_cols)}")
        print(f"- Target columns:")
        print(self.target_cols)

        print("-" * 60)

        print("KÍCH THƯỚC MỘT MẪU:")
        print(f"- X shape: {tuple(X_tr.shape)}")
        print(f"- y shape: {tuple(y_tr.shape)}")

        print("-" * 60)

        print("CHIA TẬP DỮ LIỆU:")
        print(f"1. Train:")
        print(f"   + Số dòng gốc: {len(self.df_train)}")
        print(f"   + Số cửa sổ: {len(self.train_dataset)}")
        print(f"   + Số batches: {train_batches}")

        print(f"2. Val:")
        print(f"   + Số dòng gốc: {len(self.df_val)}")
        print(f"   + Số cửa sổ: {len(self.val_dataset)}")
        print(f"   + Số batches: {val_batches}")

        print(f"3. Test:")
        print(f"   + Số dòng gốc: {len(self.df_test)}")
        print(f"   + Số cửa sổ: {len(self.test_dataset)}")
        print(f"   + Số batches: {test_batches}")

        print("=" * 60)

In [5]:
# Kiểm tra tập Appliances Energy
print("--- TEST BITCOIN ---")
app_dataset = TimeSeriesDataset(name="bitcoin", seq_len=24, pred_len=1)
app_dataset.apply_scaling("standard")
app_dataset.print_info()


--- TEST BITCOIN ---
=== THÔNG TIN DATASET: BITCOIN ===
- File path: /kaggle/input/datasets/minh3m25/time-series-dataset/bitcoin_dataset.csv
- Số dòng sau xử lý: 125877
- Chuẩn hóa: Có
- seq_len: 24
- pred_len: 1
- batch_size: 64
- num_workers: 0
------------------------------------------------------------
- Số feature: 1
- Feature columns:
['Open']
------------------------------------------------------------
- Số target: 1
- Target columns:
['Open']
------------------------------------------------------------
KÍCH THƯỚC MỘT MẪU:
- X shape: (24, 1)
- y shape: (1, 1)
------------------------------------------------------------
CHIA TẬP DỮ LIỆU:
1. Train:
   + Số dòng gốc: 88113
   + Số cửa sổ: 88089
   + Số batches: 1377
2. Val:
   + Số dòng gốc: 25176
   + Số cửa sổ: 25152
   + Số batches: 393
3. Test:
   + Số dòng gốc: 12588
   + Số cửa sổ: 12564
   + Số batches: 197


In [6]:
RANDOM_SEED = 42

SEED_WORKER = set_seed(RANDOM_SEED)

g = torch.Generator()
g.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Random seed: 42
Device: cuda


In [7]:

class InvertedEmbedding(nn.Module):
    def __init__(self, seq_len, d_model):
        super(InvertedEmbedding, self).__init__()
        self.fc1 = nn.Linear(seq_len, d_model)

    def forward(self, x):
        # x shape: (B, seq_len, N)
        x = x.permute(0, 2, 1)   # (B, N, seq_len)
        x = self.fc1(x)          # (B, N, d_model)
        return x


class EncoderBlock(nn.Module): 
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(EncoderBlock, self).__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True
        )
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout), 
            nn.Linear(d_ff, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, N, d_model)

        attention_out = self.attention(x, x, x)[0]
        x = self.norm1(x + self.dropout(attention_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x


class iTransformer(nn.Module): 
    def __init__(
        self,
        seq_len,
        pred_len,
        d_model,
        n_heads,
        d_ff,
        e_layers,
        target_indices=None,
        dropout=0.1
    ): 
        super(iTransformer, self).__init__()

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.target_indices = target_indices

        self.embedding = InvertedEmbedding(seq_len, d_model)

        self.layers = nn.ModuleList([
            EncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(e_layers)
        ])

        self.projection = nn.Linear(d_model, pred_len)

    def forward(self, x):
        # Input x shape: (B, seq_len, N)

        x = self.embedding(x)  
        # x shape: (B, N, d_model)

        for layer in self.layers:
            x = layer(x)
        # x shape: (B, N, d_model)

        x = self.projection(x)
        # x shape: (B, N, pred_len)

        x = x.permute(0, 2, 1)
        # x shape: (B, pred_len, N)

        # Nếu chỉ muốn dự đoán một số target cụ thể, ví dụ PM2.5
        if self.target_indices is not None:
            x = x[:, :, self.target_indices]
            # x shape: (B, pred_len, target_dim)

        return x

class ResidualForecastModel(nn.Module):
    def __init__(self, base_model, target_indices):
        """
        base_model:
            Model dự đoán delta/residual.

        target_indices:
            List index của các target trong feature_cols.
            Ví dụ target_cols = ["PM2.5", "PM10", "SO2"]
            thì target_indices có thể là [0, 1, 2].
        """
        super().__init__()

        self.base_model = base_model
        self.target_indices = target_indices

    def forward(self, x):
        """
        x shape:
            (B, seq_len, num_features)

        Output:
            (B, pred_len, target_dim)
        """

        # Lấy giá trị gần nhất của các target trong input window
        # last_values shape: (B, 1, target_dim)
        last_values = x[:, -1:, self.target_indices]

        # base_model dự đoán delta
        # delta shape: (B, pred_len, target_dim)
        delta = self.base_model(x)

        # Nếu pred_len = 1:
        # last_values: (B, 1, target_dim)
        # delta      : (B, 1, target_dim)
        # output     : (B, 1, target_dim)

        # Nếu pred_len > 1, PyTorch sẽ broadcast last_values theo pred_len
        y_pred = last_values + delta

        return y_pred

In [8]:
dataset = TimeSeriesDataset(
    name="bitcoin",
    seq_len=24,
    pred_len=1,
    batch_size=32,
    num_workers=2
)

dataset.apply_scaling("standard")

train_loader, val_loader, test_loader = dataset.get_loaders()

dataset.print_info()

=== THÔNG TIN DATASET: BITCOIN ===
- File path: /kaggle/input/datasets/minh3m25/time-series-dataset/bitcoin_dataset.csv
- Số dòng sau xử lý: 125877
- Chuẩn hóa: Có
- seq_len: 24
- pred_len: 1
- batch_size: 32
- num_workers: 2
------------------------------------------------------------
- Số feature: 1
- Feature columns:
['Open']
------------------------------------------------------------
- Số target: 1
- Target columns:
['Open']
------------------------------------------------------------
KÍCH THƯỚC MỘT MẪU:
- X shape: (24, 1)
- y shape: (1, 1)
------------------------------------------------------------
CHIA TẬP DỮ LIỆU:
1. Train:
   + Số dòng gốc: 88113
   + Số cửa sổ: 88089
   + Số batches: 2753
2. Val:
   + Số dòng gốc: 25176
   + Số cửa sổ: 25152
   + Số batches: 786
3. Test:
   + Số dòng gốc: 12588
   + Số cửa sổ: 12564
   + Số batches: 393


In [9]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


def wmape(y_true, y_pred, eps=1e-8):
    """
    WMAPE = sum(|y_true - y_pred|) / sum(|y_true|)
    """
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true)) + eps
    return numerator / denominator


def compute_metrics(y_true, y_pred):
    """
    Trả về MSE, MAE, RMSE, WMAPE, R2.
    """
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    wmape_score = wmape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "WMAPE": wmape_score,
        "R2": r2
    }

In [10]:
#Baseline
def evaluate_naive_baseline(dataset, loader):
    """
    Baseline univariate:
    Dự đoán PM2.5 giờ tiếp theo = PM2.5 ở giờ gần nhất trong input window.

    Yêu cầu:
    - target_cols = ["PM2.5"]
    - PM2.5 nằm trong feature_cols
    - pred_len = 1
    """

    pm25_feature_idx = dataset.feature_cols.index("PM2.5")

    all_preds = []
    all_trues = []

    for X, y in loader:
        # X shape: (batch, seq_len, num_features)
        # y shape: (batch, pred_len, target_dim)

        # Lấy PM2.5 tại timestep cuối cùng của input window
        y_pred = X[:, -1, pm25_feature_idx]

        # Vì pred_len = 1, target_dim = 1
        y_true = y[:, 0, 0]

        all_preds.append(y_pred.detach().cpu().numpy())
        all_trues.append(y_true.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_trues)

    # Nếu dataset đã chuẩn hóa, đưa về giá trị thật
    if dataset.is_scaled:
        y_pred = dataset.inverse_transform_target(
            y_pred.reshape(-1, 1, 1)
        ).reshape(-1)

        y_true = dataset.inverse_transform_target(
            y_true.reshape(-1, 1, 1)
        ).reshape(-1)

    metrics = compute_metrics(y_true, y_pred)

    print("=== NAIVE BASELINE ===")
    print("Dự đoán PM2.5 giờ tiếp theo = PM2.5 giờ gần nhất")
    print("-" * 60)
    print(f"MSE   : {metrics['MSE']:.4f}")
    print(f"MAE   : {metrics['MAE']:.4f}")
    print(f"RMSE  : {metrics['RMSE']:.4f}")
    print(f"WMAPE : {metrics['WMAPE']:.4f}")
    print(f"R2    : {metrics['R2']:.4f}")

    return metrics


def evaluate_naive_multivariate_baseline(dataset, loader):
    """
    Baseline multivariate:
    Dự đoán target(t+1) = target(t) cho tất cả target.

    Ví dụ:
    PM2.5(t+1) = PM2.5(t)
    PM10(t+1)  = PM10(t)
    SO2(t+1)   = SO2(t)
    ...
    """

    target_indices = [
        dataset.feature_cols.index(col)
        for col in dataset.target_cols
    ]

    all_preds = []
    all_trues = []

    for X, y in loader:
        # X shape: (B, seq_len, num_features)
        # y shape: (B, pred_len, target_dim)

        # Lấy giá trị gần nhất của các target
        # Shape: (B, 1, target_dim)
        y_pred = X[:, -1:, target_indices]

        # Nếu pred_len > 1, lặp baseline cho tất cả future steps
        if y.shape[1] > 1:
            y_pred = y_pred.repeat(1, y.shape[1], 1)

        all_preds.append(y_pred.detach().cpu())
        all_trues.append(y.detach().cpu())

    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_true = torch.cat(all_trues, dim=0).numpy()

    # Nếu dataset đã chuẩn hóa, đưa về giá trị thật
    if dataset.is_scaled:
        y_pred = dataset.inverse_transform_target(y_pred)
        y_true = dataset.inverse_transform_target(y_true)

    target_dim = len(dataset.target_cols)

    # Shape: (num_samples * pred_len, target_dim)
    y_pred_2d = y_pred.reshape(-1, target_dim)
    y_true_2d = y_true.reshape(-1, target_dim)

    print("=== NAIVE MULTIVARIATE BASELINE ===")
    print("Dự đoán mỗi target giờ tiếp theo = giá trị gần nhất của chính target đó")
    print("-" * 90)

    per_target_metrics = {}

    for i, col in enumerate(dataset.target_cols):
        y_t = y_true_2d[:, i]
        y_p = y_pred_2d[:, i]

        metrics = compute_metrics(y_t, y_p)
        per_target_metrics[col] = metrics

        print(
            f"{col:>8s} | "
            f"MSE: {metrics['MSE']:.4f} | "
            f"MAE: {metrics['MAE']:.4f} | "
            f"RMSE: {metrics['RMSE']:.4f} | "
            f"WMAPE: {metrics['WMAPE']:.4f} | "
            f"R2: {metrics['R2']:.4f}"
        )

    print("-" * 90)

    # Overall kiểu flatten toàn bộ target
    y_true_flat = y_true_2d.reshape(-1)
    y_pred_flat = y_pred_2d.reshape(-1)

    overall_metrics = compute_metrics(y_true_flat, y_pred_flat)

    print("=== OVERALL METRICS ===")
    print(
        f"MSE: {overall_metrics['MSE']:.4f} | "
        f"MAE: {overall_metrics['MAE']:.4f} | "
        f"RMSE: {overall_metrics['RMSE']:.4f} | "
        f"WMAPE: {overall_metrics['WMAPE']:.4f} | "
        f"R2: {overall_metrics['R2']:.4f}"
    )

    return {
        "per_target": per_target_metrics,
        "overall": overall_metrics
    }

In [11]:
#thuc hien chuan hoa du lieu de train mo hinh
dataset = TimeSeriesDataset(
    name="bitcoin",
    seq_len=24,
    pred_len=1,
    batch_size=16,
    num_workers=2
)

dataset.apply_scaling(scaler_type="standard")

train_loader, val_loader, test_loader = dataset.get_loaders()



In [12]:
# Các target cần dự đoán
target_cols = dataset.target_cols

# Index của target trong feature_cols
target_indices = [
    dataset.feature_cols.index(col)
    for col in target_cols
]

print("Target cols:", target_cols)
print("Target indices:", target_indices)

Target cols: ['Open']
Target indices: [0]


In [13]:
#Thực hiện train với toàn bộ features, sử dụng MSE loss

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred = model(X)

        if y_pred.shape != y.shape:
            raise ValueError(
                f"Shape mismatch: y_pred.shape={y_pred.shape}, y.shape={y.shape}"
            )

        loss = criterion(y_pred, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        batch_size = X.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate_loss(model, loader, criterion, device):
    model.eval()
    
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)

            loss = criterion(y_pred, y)

            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples


def evaluate_real_metrics(model, loader, dataset, device):
    """
    Tính MSE, MAE, RMSE, WMAPE, R2 trên giá trị thật sau inverse_transform.
    Phù hợp với bài toán univariate, ví dụ target_cols = ["PM2.5"].
    """

    model.eval()

    all_preds = []
    all_trues = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)

            y_pred = model(X)

            all_preds.append(y_pred.detach().cpu())
            all_trues.append(y.detach().cpu())

    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_true = torch.cat(all_trues, dim=0).numpy()

    if dataset.is_scaled:
        y_pred = dataset.inverse_transform_target(y_pred)
        y_true = dataset.inverse_transform_target(y_true)

    y_pred = y_pred.reshape(-1)
    y_true = y_true.reshape(-1)

    metrics = compute_metrics(y_true, y_pred)

    print("=== REAL METRICS ===")
    print(f"MSE   : {metrics['MSE']:.4f}")
    print(f"MAE   : {metrics['MAE']:.4f}")
    print(f"RMSE  : {metrics['RMSE']:.4f}")
    print(f"WMAPE : {metrics['WMAPE']:.4f}")
    print(f"R2    : {metrics['R2']:.4f}")

    return metrics



def evaluate_multivariate_metrics(model, loader, dataset, device):
    """
    Tính MSE, MAE, RMSE, WMAPE, R2 cho bài toán multivariate forecasting.

    Trả về:
    - overall metrics
    - per-target metrics
    """

    model.eval()

    all_preds = []
    all_trues = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)

            y_pred = model(X)

            all_preds.append(y_pred.detach().cpu())
            all_trues.append(y.detach().cpu())

    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_true = torch.cat(all_trues, dim=0).numpy()

    if dataset.is_scaled:
        y_pred = dataset.inverse_transform_target(y_pred)
        y_true = dataset.inverse_transform_target(y_true)

    target_dim = len(dataset.target_cols)

    # Shape: (num_samples * pred_len, target_dim)
    y_pred_2d = y_pred.reshape(-1, target_dim)
    y_true_2d = y_true.reshape(-1, target_dim)

    print("=== MULTIVARIATE METRICS PER TARGET ===")
    print("-" * 100)

    per_target_metrics = {}

    for i, col in enumerate(dataset.target_cols):
        y_t = y_true_2d[:, i]
        y_p = y_pred_2d[:, i]

        metrics = compute_metrics(y_t, y_p)
        per_target_metrics[col] = metrics

        print(
            f"{col:>8s} | "
            f"MSE: {metrics['MSE']:.4f} | "
            f"MAE: {metrics['MAE']:.4f} | "
            f"RMSE: {metrics['RMSE']:.4f} | "
            f"WMAPE: {metrics['WMAPE']:.4f} | "
            f"R2: {metrics['R2']:.4f}"
        )

    print("-" * 100)

    # Overall dạng flatten toàn bộ target
    # Lưu ý: metric này có thể bị CO chi phối nếu CO có scale lớn.
    y_true_flat = y_true_2d.reshape(-1)
    y_pred_flat = y_pred_2d.reshape(-1)

    overall_metrics = compute_metrics(y_true_flat, y_pred_flat)

    print("=== OVERALL METRICS ===")
    print(
        f"MSE: {overall_metrics['MSE']:.4f} | "
        f"MAE: {overall_metrics['MAE']:.4f} | "
        f"RMSE: {overall_metrics['RMSE']:.4f} | "
        f"WMAPE: {overall_metrics['WMAPE']:.4f} | "
        f"R2: {overall_metrics['R2']:.4f}"
    )

    return {
        "overall": overall_metrics,
        "per_target": per_target_metrics
    }

In [14]:
# danh gia baseline
train_loader, val_loader, test_loader = dataset.get_loaders()
print("Đánh giá trên tập val")
evaluate_naive_multivariate_baseline(dataset, val_loader)

Đánh giá trên tập val
=== NAIVE MULTIVARIATE BASELINE ===
Dự đoán mỗi target giờ tiếp theo = giá trị gần nhất của chính target đó
------------------------------------------------------------------------------------------
    Open | MSE: 60126.9805 | MAE: 140.9869 | RMSE: 245.2080 | WMAPE: 0.0036 | R2: 0.9998
------------------------------------------------------------------------------------------
=== OVERALL METRICS ===
MSE: 60126.9805 | MAE: 140.9869 | RMSE: 245.2080 | WMAPE: 0.0036 | R2: 0.9998


{'per_target': {'Open': {'MSE': 60126.98046875,
   'MAE': 140.9868927001953,
   'RMSE': np.float64(245.2080350819483),
   'WMAPE': np.float32(0.0035781467),
   'R2': 0.9998313188552856}},
 'overall': {'MSE': 60126.98046875,
  'MAE': 140.9868927001953,
  'RMSE': np.float64(245.2080350819483),
  'WMAPE': np.float32(0.0035781467),
  'R2': 0.9998313188552856}}

In [15]:

# Định nghĩa các tham số (Hyperparameters)
d_model = 128     # Thử 64 trước, nếu underfit thì lên 128
n_heads = 8         
d_ff = 256          # Giữ ở mức 2 * d_model
e_layers = 2        # Thử 1 lớp trước, tối đa là 2
dropout = 0.2

base_model = iTransformer(
    seq_len=dataset.seq_len,
    pred_len=dataset.pred_len,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    e_layers=e_layers,
    target_indices=target_indices,
    dropout=dropout
).to(DEVICE)

model = ResidualForecastModel(
    base_model=base_model,
    target_indices=target_indices
).to(DEVICE)

#Thực hiện train
for X, y in train_loader:
    X = X.to(DEVICE)
    y = y.to(DEVICE)

    y_pred = model(X)

    print("X shape     :", X.shape)
    print("y shape     :", y.shape)
    print("y_pred shape:", y_pred.shape)

    break
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,          # Learning rate lớn hơn một chút vì mô hình nhỏ
    weight_decay=1e-3 
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

num_epochs = 80
patience = 7

best_val_loss = float("inf")
best_model_state = None
patience_counter = 0

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )

    val_loss = evaluate_loss(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )
    
    scheduler.step(val_loss)
    
    val_metrics = evaluate_multivariate_metrics(
        model=model,
        loader=val_loader,
        dataset=dataset,
        device=DEVICE
    )

    val_overall = val_metrics["overall"]

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch:03d}/{num_epochs}] "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Val MSE: {val_overall['MSE']:.4f} | "
        f"Val MAE: {val_overall['MAE']:.4f} | "
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping.")
        break

model.load_state_dict(best_model_state)
print("Đã hoàn thành train")




X shape     : torch.Size([16, 24, 1])
y shape     : torch.Size([16, 1, 1])
y_pred shape: torch.Size([16, 1, 1])
=== MULTIVARIATE METRICS PER TARGET ===
----------------------------------------------------------------------------------------------------
    Open | MSE: 61060.9453 | MAE: 143.5184 | RMSE: 247.1051 | WMAPE: 0.0036 | R2: 0.9998
----------------------------------------------------------------------------------------------------
=== OVERALL METRICS ===
MSE: 61060.9453 | MAE: 143.5184 | RMSE: 247.1051 | WMAPE: 0.0036 | R2: 0.9998
Epoch [001/80] Train Loss: 0.011143 | Val Loss: 0.000295 | Val MSE: 61060.9453 | Val MAE: 143.5184 | 
=== MULTIVARIATE METRICS PER TARGET ===
----------------------------------------------------------------------------------------------------
    Open | MSE: 75840.1953 | MAE: 190.7921 | RMSE: 275.3910 | WMAPE: 0.0048 | R2: 0.9998
----------------------------------------------------------------------------------------------------
=== OVERALL METRICS ==

In [16]:
print("Đánh giá trên tập test")

baseline_metrics = evaluate_naive_multivariate_baseline(
    dataset=dataset,
    loader=test_loader
)

model_metrics = evaluate_multivariate_metrics(
    model=model,
    loader=test_loader,
    dataset=dataset,
    device=DEVICE
)

baseline_overall = baseline_metrics["overall"]
model_overall = model_metrics["overall"]

print("\n=== SO SÁNH OVERALL ===")
print(f"Baseline MAE   : {baseline_overall['MAE']:.4f}")
print(f"Model MAE      : {model_overall['MAE']:.4f}")

print(f"Baseline MSE   : {baseline_overall['MSE']:.4f}")
print(f"Model MSE      : {model_overall['MSE']:.4f}")

print(f"Baseline RMSE  : {baseline_overall['RMSE']:.4f}")
print(f"Model RMSE     : {model_overall['RMSE']:.4f}")

print(f"Baseline WMAPE : {baseline_overall['WMAPE']:.4f}")
print(f"Model WMAPE    : {model_overall['WMAPE']:.4f}")

print(f"Baseline R2    : {baseline_overall['R2']:.4f}")
print(f"Model R2       : {model_overall['R2']:.4f}")

Đánh giá trên tập test
=== NAIVE MULTIVARIATE BASELINE ===
Dự đoán mỗi target giờ tiếp theo = giá trị gần nhất của chính target đó
------------------------------------------------------------------------------------------
    Open | MSE: 201687.8594 | MAE: 296.2966 | RMSE: 449.0967 | WMAPE: 0.0031 | R2: 0.9992
------------------------------------------------------------------------------------------
=== OVERALL METRICS ===
MSE: 201687.8594 | MAE: 296.2966 | RMSE: 449.0967 | WMAPE: 0.0031 | R2: 0.9992
=== MULTIVARIATE METRICS PER TARGET ===
----------------------------------------------------------------------------------------------------
    Open | MSE: 201745.1562 | MAE: 296.2870 | RMSE: 449.1605 | WMAPE: 0.0031 | R2: 0.9992
----------------------------------------------------------------------------------------------------
=== OVERALL METRICS ===
MSE: 201745.1562 | MAE: 296.2870 | RMSE: 449.1605 | WMAPE: 0.0031 | R2: 0.9992

=== SO SÁNH OVERALL ===
Baseline MAE   : 296.2966
Model MA